# 09b. 골프장·레고랜드 좌표 추가

## 왜 하나
- 상가(상권)정보에는 **실외 골프장이 없다** (춘천 야외레저 매출의 핵심인데 위치가 빠짐)
- 강원도 공공체육시설 목록(2020)의 춘천 골프장 11곳 + 이후 개장한 베어크리크 춘천 1곳 + 레고랜드 1곳 = **13곳**의 좌표를 직접 수집해 추가

## 방법
1. `chuncheon_golf_legoland_coordinates_v3.csv` 읽기 (위도·경도, WGS84)
2. 상가 데이터와 같은 좌표계(EPSG:5179)로 변환
3. `출처` 컬럼으로 상가정보와 수동 추가를 구분해서 합치기
4. 저장: `stores_mapped_with_golf.csv`

## 주의
- 골프장은 한 점이 아니라 넓은 땅이다 → 격자 분석에서 가게 1곳처럼 세지 말고 "골프장 있음" 표시로 따로 쓴다
- 레고랜드 좌표는 v2에서 오류(춘천 밖 약 35km)를 발견해 주소(하중도길 128) 기준으로 v3에서 수정했다

# 셀2 (코드) 준비

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q pyproj

from pathlib import Path
import pandas as pd
from pyproj import Transformer

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data" / "processed"

셀 3 (코드): 좌표 추가 + 저장

In [ ]:
# 1. 좌표 파일 읽기
golf = pd.read_csv(OUT / "chuncheon_golf_legoland_coordinates_v3.csv")
print("좌표 파일:", len(golf), "곳")

# 2. 위도·경도(WGS84) → EPSG:5179 (상가 데이터와 같은 좌표계)
tf = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
golf["x"], golf["y"] = tf.transform(golf["longitude"].values, golf["latitude"].values)

# 3. 상가 데이터 읽기
stores = pd.read_csv(OUT / "stores_mapped.csv")

# 4. 상가 데이터 형식에 맞춰 추가할 행 만들기
add = pd.DataFrame({
    "상호명": golf["facility_name"],
    "시군구명": "춘천시",
    "경도": golf["longitude"],
    "위도": golf["latitude"],
    "region": "춘천",
    "x": golf["x"],
    "y": golf["y"],
    "ind_grp": "야외레저",
    "출처": "수동추가_" + golf["facility_type"],
})

# 5. 합치기 + 저장
stores["출처"] = "상가정보"
merged = pd.concat([stores, add], ignore_index=True)
merged.to_csv(OUT / "stores_mapped_with_golf.csv", index=False, encoding="utf-8-sig")
print("저장 완료: stores_mapped_with_golf.csv")

# 셀4 (코드) : 확인


In [ ]:
print("추가:", len(add), "곳 | 전체:", f"{len(stores):,}", "→", f"{len(merged):,}")
print(merged["출처"].value_counts())
print(add[["상호명", "출처", "위도", "경도", "x", "y"]].round(4).to_string(index=False))

# 춘천 상가 좌표 범위 안에 들어오는지 확인
ch = stores[stores["region"] == "춘천"]
inside = add["x"].between(ch["x"].min(), ch["x"].max()) & add["y"].between(ch["y"].min(), ch["y"].max())
print("\n춘천 상가 범위 안:", inside.sum(), "/", len(add))